In [ ]:
import kaggle_benchmarks as kbench

print('Available benchmark models:')
for slug in sorted(kbench.llms.keys()):
    print(f'  "{slug}"')

In [ ]:
# --------------------------------------------------------------------------------
# 📚 LEARNING RESOURCES
# Quick Start: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/quick_start.md
# Cookbook:    https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
# --------------------------------------------------------------------------------

# ── Models to evaluate ────────────────────────────────────────────────
MODELS = [
    kbench.llms["google/gemini-2.5-flash"],
    kbench.llms["google/gemini-3.1-pro-preview"],
]

# --------------------------------------------------------------------------------
# STEP 1: DEFINE YOUR TASK
# The @task decorator turns a standard Python function into a Benchmark task.
# The first parameter must always be `llm` (the model being tested).
# --------------------------------------------------------------------------------
@kbench.task(name="What is Kaggle?", description="Does the LLM know what Kaggle is?")
def what_is_kaggle(llm) -> None:

    # A. Prompt the model
    response: str = llm.prompt("What is Kaggle?")

    # B. Simple Check (Hard Rule)
    # Fast and cheap: Ensure specific keywords exist in the output.
    kbench.assertions.assert_in("platform", response.lower())

    # C. Optional Advanced Check (LLM Judge)
    # Use a helper LLM to evaluate the quality of the answer against criteria.
    assessment = kbench.assertions.assess_response_with_judge(
        response_text=response,
        judge_llm=kbench.judge_llm,
        criteria=[
            "The answer must mention data science or machine learning.",
            "The answer should mention competitions.",
        ]
    )

    # Iterate through the judge's feedback and assert success
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Judge Criterion '{result.criterion}' should pass: {result.reason}"
        )

# --------------------------------------------------------------------------------
# STEP 2: RUN THE TASK
# Run all models in one session → one .run.json per model, all retrievable at once.
# To test a single model via the Kaggle UI "Add Models" button, use kbench.llm:
#   what_is_kaggle.run(kbench.llm)
# --------------------------------------------------------------------------------
for model in MODELS:
    what_is_kaggle.run(model)

# --------------------------------------------------------------------------------
# STEP 3: NEXT STEPS
# 1. Click "Save Task" (top right) to publish to the leaderboard.
# 2. Try `%autopilot` in a new cell to auto-generate tasks or write your own!
# --------------------------------------------------------------------------------